# UNDERSTANDING THE DATA
This Python Notebook shows analysis steps related to the dataset: data fetching, cleaning, EDA, clustering and relevant charts.

## STEP 1. Downloading local .csv file
Due to ISTAT's request limitations, I will download the dataset as a local .csv file, as to not exceed ISTAT's request limit.

In [5]:
import requests
from pathlib import Path

url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
headers = {"Accept": "application/vnd.sdmx.data+csv;version=1.0.0"}
path = Path("data.csv")  # nella stessa cartella del notebook

r = requests.get(url, headers=headers, verify=False)
path.write_bytes(r.content)

print(f"File downloaded.")

c:\Users\Mauro\OneDrive\Desktop\final-project\final-project\.venv\Lib\site-packages\urllib3\connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'esploradati.istat.it'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File downloaded.


## STEP 2. Loading the .csv file and analyzing


In [6]:
import pandas as pd

data = pd.read_csv(path, dtype=str)
print(data.shape)
data.head()

(573552, 16)


,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,001001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,001001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,001001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,001001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,001001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The file has 573.552 rows and 16 columns.
The columns are:
- DATAFLOW
- FREQ
- REF_AREA
- DATA_TYPE
- RESULT
- TIME_PERIOD
- OBS_VALUE
- OBS_STATUS
- NOTE_DS
- NOTE_REF_AREA
- NOTE_DATA_TYPE
- NOTE_RESULT
- NOTE_TIME_PERIOD
- BASE_PER
- UNIT_MEAS
- UNIT_MULT

### DATAFLOW
The "DATAFLOW" column has the same value for all rows.

In [7]:
print(data["DATAFLOW"].unique())


<StringArray>
['IT1:41_983(1.0)']
Length: 1, dtype: str


This means that the column can be omitted completely from the dataset.

### FREQ
The "FREQ" column too has the same value for all rows.

In [8]:
print(data["FREQ"].unique())

<StringArray>
['A']
Length: 1, dtype: str


This column can be omitted as well.

### REF_AREA
This columns contains 8578 unique values, which correspond to the ISTAT codes of Italian cities (e.g. 001001 is the city of Agliè, in the Italian province of Turin).

In [9]:
print(data["REF_AREA"].unique())

<StringArray>
['001001', '001002', '001003', '001004', '001005', '001006', '001007',
 '001008', '001009', '001010',
 ...
 '111098', '111099', '111100', '111101', '111102', '111103', '111104',
 '111105', '111106', '111107']
Length: 8578, dtype: str


Each value repeats across multiple rows, which means that that specific city has had multiple accidents in the same or different years.

### DATA_TYPE
This column contains two unique values: KILLINGJ and ROADACC. It probably explains what a different column in the same row refers to (e.g. if this column has "ROADACC" and another column has "5", it might mean that 5 road accidents have happened in that city in that year; if this column has "KILLINGJ" and another column has "2", it might mean that 2 people have been killed or injured in one of those accidents).

In [10]:
print(data["DATA_TYPE"].unique())

<StringArray>
['KILLINJ', 'ROADACC']
Length: 2, dtype: str


### RESULT
The RESULT column can have three distinct values: "F", "M", and "9".

In [11]:
print(data["RESULT"].unique())

<StringArray>
['F', 'M', '9']
Length: 3, dtype: str


In particular, RESULT has a value of "9" only when DATA_TYPE is "ROADACC", whereas it can have a value of either "F" or "M" when DATA_TYPE is "KILLINJ".

In [13]:
data[data["DATA_TYPE"] == "KILLINJ"]["RESULT"].unique()


<StringArray>
['F', 'M']
Length: 2, dtype: str

In [14]:
data[data["DATA_TYPE"] == "ROADACC"]["RESULT"].unique()

<StringArray>
['9']
Length: 1, dtype: str

This can only mean that:
- a combination of "KILLINJ" in DATA_TYPE and "M" in RESULT means that another column specifies the number of deaths ("M" stands for "MORTI");
- a combination of "KILLINJ" in DATA_TYPE and "F" in RESULT means that another column specifies the number of injuries ("F" stands for "FERITI");
- a combination of "ROADACC" in DATA_TYPE and "9" in RESULT means that another column specifies the number of total road accidents (not sure whar "9" stands for, might just be a placeholder).

### TIME_PERIOD
This column contains 24 distinct years, from 2001 to 2024.

In [15]:
print(data["TIME_PERIOD"].unique())

<StringArray>
['2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009',
 '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018',
 '2019', '2020', '2021', '2022', '2023', '2024']
Length: 24, dtype: str


### OBS_VALUE
This column contains numeric values. It's never negative (the minimum value is zero) or empty (there are no null values).

In [20]:
print("Minimum value:", data["OBS_VALUE"].min())
print("Missing values:", data["OBS_VALUE"].isna().sum())

Minimum value: 0
Missing values: 0


This might be the number that counts deaths, injuries, and road accidents (the combination of DATA_TYPE and RESULT).

### OBS_STATUS, NOTE_DS, NOTE_REF_AREA, NOTE_DATA_TYPE, NOTE_RESULT, NOTE_TIME_PERIOD, BASE_PER, UNIT_MEAS, UNIT_MULT
All these columns are empty: the only value is nan.

In [23]:
print(data["OBS_STATUS"].unique())
print(data["NOTE_DS"].unique())
print(data["NOTE_REF_AREA"].unique())
print(data["NOTE_DATA_TYPE"].unique())
print(data["NOTE_RESULT"].unique())
print(data["NOTE_TIME_PERIOD"].unique())
print(data["BASE_PER"].unique())
print(data["UNIT_MEAS"].unique())
print(data["UNIT_MULT"].unique())

<StringArray>
[nan]
Length: 1, dtype: str
<StringArray>
[nan]
Length: 1, dtype: str
<StringArray>
[nan]
Length: 1, dtype: str
<StringArray>
[nan]
Length: 1, dtype: str
<StringArray>
[nan]
Length: 1, dtype: str
<StringArray>
[nan]
Length: 1, dtype: str
<StringArray>
[nan]
Length: 1, dtype: str
<StringArray>
[nan]
Length: 1, dtype: str
<StringArray>
[nan]
Length: 1, dtype: str


## STEP 3. Conclusions

**Columns to omit**. Columns DATAFLOW and FREQ have the same value in all rows, whereas columns OBS_STATUS, NOTE_DS, NOTE_REF_AREA, NOTE_DATA_TYPE, NOTE_RESULT, NOTE_TIME_PERIOD, BASE_PER, UNIT_MEAS, and UNIT_MULT are empty; all these columns can thus be omitted from the final dataframe.

**Interaction between DATA_TYPE and RESULT**. The DATA_TYPE and RESULT columns specify what the value of column OBS_VALUE means. This entails that the columns can be synthesized into one single column that specifies whether OBS_VALUE is the number of deaths, the number of injuries, or the number of accidents.

**Columns in the final dataframe**. The final dataframe, not including data from other sources, will contain four columns:
- REF_AREA: ISTAT code of each municipality;
- TIME_PERIOD: year of the observation;
- OBSERVATION: what OBS_VALUE counts (accidents, injuries or deaths), obtained by merging DATA_TYPE and RESULT;
- OBS_VALUE: the numeric count of observation.

Each row will represent one observation in a municipality in a year.